# Benchmark Extract - Entity Extraction với DeepSeek API

Notebook này sử dụng DeepSeek API để trích xuất thực thể từ sách giáo khoa lịch sử.

## Chế độ xử lý:
- **JSON Hierarchical** (mặc định): Xử lý theo cấu trúc semantic (Chủ đề → Bài → Section → Subsection)
- **TXT Legacy**: Xử lý theo windows cố định từ file text

In [1]:
# Cell 1: Nhập API Key (được che khi nhập)
import os
import sys
from getpass import getpass

# Thêm extract folder vào path
sys.path.insert(0, 'extract')

# Kiểm tra API key từ environment trước
api_key = os.environ.get('DEEPSEEK_API_KEY', '')

if not api_key:
    print('DEEPSEEK_API_KEY không tìm thấy trong environment.')
    api_key = getpass('Nhập DeepSeek API Key (ẩn khi nhập): ')
    
    if api_key:
        os.environ['DEEPSEEK_API_KEY'] = api_key
        print(f'API Key đã được set: {api_key[:8]}...{api_key[-4:]}')
    else:
        print('ERROR: Không có API key!')
else:
    print(f'API Key từ environment: {api_key[:8]}...{api_key[-4:]}')

print('Ready!')

DEEPSEEK_API_KEY không tìm thấy trong environment.


Nhập DeepSeek API Key (ẩn khi nhập):  ········


API Key đã được set: sk-edcb3...e2d1
Ready!


In [2]:
# Cell 2: Import và kiểm tra config
import config

print('=== CẤU HÌNH ===')
print(f'USE_JSON_FORMAT: {getattr(config, "USE_JSON_FORMAT", False)}')
print(f'JSON_INPUT_FILE: {getattr(config, "JSON_INPUT_FILE", "N/A")}')
print(f'DEEPSEEK_MODEL: {getattr(config, "DEEPSEEK_MODEL", "deepseek-chat")}')
print(f'API_DELAY_SECONDS: {config.API_DELAY_SECONDS}')
print(f'WINDOW_SIZE: {config.WINDOW_SIZE}')

# Kiểm tra JSON file
json_file = getattr(config, 'JSON_INPUT_FILE', None)
if json_file and os.path.exists(json_file):
    print(f'\n✓ JSON file exists: {json_file}')
else:
    print(f'\n✗ JSON file NOT found: {json_file}')

=== CẤU HÌNH ===
USE_JSON_FORMAT: True
JSON_INPUT_FILE: SGK\SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json
DEEPSEEK_MODEL: deepseek-chat
API_DELAY_SECONDS: 2
WINDOW_SIZE: 5

✓ JSON file exists: SGK\SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json


In [3]:
# Cell 3: Test DeepSeek API
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)

print('Testing DeepSeek API connection...')
response = client.chat.completions.create(
    model='deepseek-chat',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant'},
        {'role': 'user', 'content': 'Reply OK if you can read this'}
    ],
    stream=False
)

print(f'Response: {response.choices[0].message.content}')
print('✓ DeepSeek API OK!')

Testing DeepSeek API connection...
Response: OK
✓ DeepSeek API OK!


In [4]:
# Cell 4: Xem thống kê JSON (nếu dùng JSON mode)
if getattr(config, 'USE_JSON_FORMAT', False):
    try:
        from json_processor import JSONTextbookProcessor
        
        processor = JSONTextbookProcessor(config.JSON_INPUT_FILE)
        stats = processor.get_statistics()
        
        print('=== THỐNG KÊ JSON SÁCH GIÁO KHOA ===')
        print(f'Tổng chunks (semantic units): {stats["total_chunks"]}')
        print(f'Tổng chủ đề: {stats["total_topics"]}')
        print(f'Tổng bài: {stats["total_lessons"]}')
        print(f'Tổng câu: {stats["total_sentences"]}')
        print(f'Trung bình câu/chunk: {stats["avg_sentences_per_chunk"]:.1f}')
        
        print('\n=== MẪU CHUNKS ===')
        for i, chunk in enumerate(list(processor.iter_chunks())[:3]):
            print(f'\n[{i+1}] {chunk.full_path}')
            print(f'    Section: {chunk.section_title}')
            print(f'    Subsection: {chunk.subsection_title}')
            print(f'    Sentences: {len(chunk.content)}')
    except Exception as e:
        print(f'Error loading JSON: {e}')
else:
    print('TXT mode - skipping JSON stats')

=== THỐNG KÊ JSON SÁCH GIÁO KHOA ===
Tổng chunks (semantic units): 76
Tổng chủ đề: 6
Tổng bài: 17
Tổng câu: 782
Trung bình câu/chunk: 10.3

=== MẪU CHUNKS ===

[1] Chủ đề 1/Bài 1/Section 1/a
    Section: Một số vấn đề cơ bản về Liên hợp quốc
    Subsection: Bối cảnh lịch sử và quá trình hình thành
    Sentences: 8

[2] Chủ đề 1/Bài 1/Section 1/b
    Section: Một số vấn đề cơ bản về Liên hợp quốc
    Subsection: Mục tiêu, nguyên tắc hoạt động
    Sentences: 9

[3] Chủ đề 1/Bài 1/Section 2/a
    Section: Vai trò của Liên hợp quốc
    Subsection: Duy trì hoà bình, an ninh quốc tế
    Sentences: 5


In [5]:
# Cell 5: Import các modules cần thiết
import text_processor
import api_handler
import entity_processor
import output_manager
from api_handler import reset_request_counter

# Kiểm tra json_processor có sẵn không
try:
    from json_processor import JSONTextbookProcessor
    print('✓ JSON Hierarchical Processor available')
except ImportError:
    print('✗ JSON Processor not available, will use TXT mode')

print('\nAll modules loaded successfully!')

✓ JSON Hierarchical Processor available

All modules loaded successfully!


In [6]:
# Cell 6: Run entity extraction
from main import process_entities

print('=' * 60)
print('STARTING ENTITY EXTRACTION')
print('=' * 60)

# Run extraction
entities = process_entities()

print('\n' + '=' * 60)
print('EXTRACTION COMPLETED')
print('=' * 60)
print(f'Total entities: {len(entities)}')

STARTING ENTITY EXTRACTION
[MODE] JSON Hierarchical Processing (semantic chunking)

[JSON HIERARCHICAL MODE]
Loading from: SGK\SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json

Thống kê dữ liệu:
  - Tổng chunks (subsections): 76
  - Tổng chủ đề: 6
  - Tổng bài: 17
  - Tổng câu: 782
  - Trung bình câu/chunk: 10.3
   [Split] Chủ đề 1/Bài 1/Section 1/a: 8 sentences, 1856 chars -> 4 parts
   [Split] Chủ đề 1/Bài 1/Section 1/b: 9 sentences, 1557 chars -> 3 parts
   [Split] Chủ đề 1/Bài 1/Section 2/b: 7 sentences, 1526 chars -> 4 parts
   [Split] Chủ đề 1/Bài 2/Section 1/a: 13 sentences, 2085 chars -> 4 parts
   [Split] Chủ đề 1/Bài 2/Section 1/b: 18 sentences, 3020 chars -> 4 parts
   [Split] Chủ đề 1/Bài 3/Section 2/b: 11 sentences, 1823 chars -> 4 parts
   [Split] Chủ đề 2/Bài 4/Section 2/b: 4 sentences, 1565 chars -> 4 parts
   [Split] Chủ đề 2/Bài 5/Section 3/: 10 sentences, 2094 chars -> 4 parts
   [Split] Chủ đề 3/Bài 6/Section 1/: 11 sentences, 1614 chars -> 4 parts
   [Split] Chủ đề 3/Bài 6/Sec

In [7]:
# Cell 7: Xem kết quả và lưu
# Thống kê theo loại
type_counts = {}
for entity in entities:
    t = entity.get('type', 'Unknown')
    type_counts[t] = type_counts.get(t, 0) + 1

print('=== THỐNG KÊ THEO LOẠI ===')
for t, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t}: {count}')

# Lưu kết quả
output_manager.save_entities(entities)
print('\n✓ Entities saved!')

=== THỐNG KÊ THEO LOẠI ===
  Địa điểm: 147
  Tổ chức: 103
  Khái niệm: 86
  Văn kiện/Hiệp định: 59
  Sự kiện: 50
  Quốc gia: 44
  Chiến lược/Chủ trương: 36
  Hội nghị: 30
  Chiến dịch/Trận đánh: 27
  Nhân Vật: 25
  Công trình: 18

=== REQUEST STATISTICS ===
Total API Requests: 195
Successful: 195 (100.0%)
Failed: 0
Total Processing Time: 4585.02s
Average per Request: 23.51s
Total Entities Extracted: 0
Total Entities Processed: 1301
Final Unique Entities: 625
Statistics saved to: entities/request_statistics_20251210_103252.json

=== REQUESTS BY SOURCE ===
Một số vấn đề cơ bản về Liên hợp quốc: 7 requests, 23 entities
Vai trò của Liên hợp quốc: 6 requests, 17 entities
Quá trình hình thành và tồn tại của Trật tự thế giới hai cực I-an-ta: 8 requests, 76 entities
Nguyên nhân và tác động của sự sụp đổ Trật tự thế giới hai cực I-an-ta: 2 requests, 9 entities
Các xu thế phát triển chính của thế giới sau chiến tranh lạnh: 1 requests, 5 entities
Xu thế đa cực trong quan hệ quốc tế: 5 requests, 2

In [9]:
# Cell 8: Xem một số entities mẫu
print('=== MẪU ENTITIES ===')
for entity in entities[:10]:
    print(f"\n {entity['id']}")
    print(f"   Type: {entity['type']}")
    print(f"   Labels: {entity.get('label', [])}")
    if entity.get('metadata'):
        print(f"   Source: {entity['metadata'].get('topic', '')} / {entity['metadata'].get('lesson', '')}")

=== MẪU ENTITIES ===

 Chiến tranh thế giới thứ hai
   Type: Sự kiện
   Labels: ['Chiến tranh thế giới thứ hai']
   Source: Chủ đề 1 / Bài 1

 Hội Quốc liên
   Type: Tổ chức
   Labels: ['Hội Quốc liên']
   Source: Chủ đề 1 / Bài 1

 Liên Xô
   Type: Quốc gia
   Labels: ['Liên Xô', 'Liên bang Xô viết']
   Source: Chủ đề 1 / Bài 1

 Liên hợp quốc
   Type: Tổ chức
   Labels: ['Liên hợp quốc']
   Source: Chủ đề 1 / Bài 1

 Tuyên bố về Liên hợp quốc
   Type: Văn kiện/Hiệp định
   Labels: ['Tuyên bố về Liên hợp quốc', 'bản Tuyên bố về Liên hợp quốc']
   Source: Chủ đề 1 / Bài 1

 Hội nghị Tê-hê-ran
   Type: Hội nghị
   Labels: ['Hội nghị Tê-hê-ran']
   Source: Chủ đề 1 / Bài 1

 I-ran
   Type: Quốc gia
   Labels: ['I-ran']
   Source: Chủ đề 1 / Bài 1

 Hội nghị I-an-ta
   Type: Hội nghị
   Labels: ['Hội nghị I-an-ta']
   Source: Chủ đề 1 / Bài 1

 Hiến chương Liên hợp quốc
   Type: Văn kiện/Hiệp định
   Labels: ['Hiến chương Liên hợp quốc']
   Source: Chủ đề 1 / Bài 1

 Xan Phran-xi-xcô
   T